# Этап 4. EDA

Вход: `data/features.csv` (этап 3)

Выход: графики в `figures/`

In [ ]:
%pip install -q "yt-dlp[default]" imageio-ffmpeg rapidfuzz librosa soundfile umap-learn transformers tqdm
%pip install -q deno

In [ ]:
import os, sys, json, time, re, subprocess, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 80)

IN_COLAB = 'google.colab' in sys.modules
USE_DRIVE = True

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/music_dataset')
else:
    BASE_DIR = Path('music_dataset').resolve()

AUDIO_DIR, DATA_DIR, FIG_DIR = BASE_DIR / 'audio', BASE_DIR / 'data', BASE_DIR / 'figures'
for d in (AUDIO_DIR, DATA_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)

GENRE_ORDER = ['rock', 'pop', 'hip_hop', 'electronic', 'jazz', 'classical',
               'metal', 'rnb_soul', 'country', 'reggae', 'ambient', 'latin']
GENRE_RU = {'rock': 'рок', 'pop': 'поп', 'hip_hop': 'хип-хоп', 'electronic': 'электроника',
            'jazz': 'джаз', 'classical': 'классика', 'metal': 'метал', 'rnb_soul': 'R&B / соул',
            'country': 'кантри', 'reggae': 'регги', 'ambient': 'эмбиент', 'latin': 'латино'}

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

CAT = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#6250d6', '#e34948']
ACCENT = CAT[0]
sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 150, 'axes.titleweight': 'bold',
                     'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False})

# жанров 12, а цветов 8, поэтому последние 4 жанра рисую треугольниками
def genre_style(g):
    i = GENRE_ORDER.index(g)
    return CAT[i % 8], ('o' if i < 8 else '^')

def savefig(name):
    plt.savefig(FIG_DIR / f'{name}.png', bbox_inches='tight')

def scatter_by_genre(ax, xy, genres, title):
    for g in GENRE_ORDER:
        m = np.asarray(genres) == g
        if m.any():
            c, mk = genre_style(g)
            ax.scatter(xy[m, 0], xy[m, 1], s=46, c=c, marker=mk, label=GENRE_RU[g],
                       edgecolors='white', linewidths=1)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)

## 4. EDA
Смотрю на состав датасета, распределения тегов, связь тегов с жанрами, сигнальные признаки по жанрам и корреляции. Отдельно проверяю, как CLAP угадывает жанр. Графики сохраняются в `figures/`.

In [ ]:
features = pd.read_csv(DATA_DIR / 'features.csv')
MOODS = [c[5:] for c in features.columns if c.startswith('mood_') and not c.startswith(('mood_top'))]
CTXS = [c[4:] for c in features.columns if c.startswith('ctx_') and not c.startswith('ctx_top')]
ATMS = [c[4:] for c in features.columns if c.startswith('atm_') and not c.startswith('atm_top')]
INSTS = [c[5:] for c in features.columns if c.startswith('inst_')]
SIGNAL = ['tempo_bpm', 'rms_mean', 'rms_dynamic_range_db', 'spectral_centroid_hz', 'spectral_flatness',
          'onset_rate', 'pulse_clarity', 'percussive_ratio', 'zcr_mean', 'mode_major', 'key_strength']
MOOD_RU = {'energetic': 'энергичное', 'calm': 'спокойное', 'happy': 'радостное', 'sad': 'грустное',
           'aggressive': 'агрессивное', 'romantic': 'романтичное', 'dreamy': 'мечтательное', 'dark': 'мрачное'}
CTX_RU = {'workout': 'тренировка', 'party': 'вечеринка', 'focus_work': 'работа / учёба', 'relaxation': 'отдых',
          'sleep': 'сон', 'road_trip': 'в дорогу', 'romantic_evening': 'романтический вечер'}
ATM_RU = {'epic': 'эпичная', 'intimate': 'камерная', 'groovy': 'грувовая', 'mysterious': 'загадочная',
          'nostalgic': 'ностальгическая', 'festive': 'праздничная'}
INST_RU = {'electric_guitar': 'электрогитара', 'acoustic_guitar': 'акуст. гитара', 'piano': 'фортепиано',
           'synthesizer': 'синтезатор', 'strings': 'струнные', 'brass_sax': 'духовые / сакс', 'drums': 'ударные'}
genre_order = [g for g in GENRE_ORDER if g in set(features.genre)]
features.shape

### 4.1 Состав

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
cnt = features['genre'].value_counts().reindex(genre_order)
axes[0].barh([GENRE_RU[g] for g in genre_order][::-1], cnt.values[::-1], color=ACCENT, height=0.6)
axes[0].set_title('Треков по жанрам'); axes[0].set_xlabel('треков')

dec = features['decade'].value_counts().sort_index()
axes[1].bar(dec.index.astype(str), dec.values, color=ACCENT, width=0.7)
axes[1].set_title('Треков по десятилетиям'); axes[1].tick_params(axis='x', rotation=60)

d = features[features.genre != 'classical']
for g in genre_order:
    if g == 'classical':
        continue
    c, mk = genre_style(g)
    m = d.genre == g
    axes[2].scatter(d.loc[m, 'year'], d.loc[m, 'log_views'], c=c, marker=mk, s=40, label=GENRE_RU[g],
                    edgecolors='#fcfcfb', linewidths=1)
axes[2].set_title('Популярность и год (без классики)'); axes[2].set_xlabel('год'); axes[2].set_ylabel('log10(просмотры)')
axes[2].legend(fontsize=7, ncol=2, loc='lower right')
plt.tight_layout(); savefig('01_dataset_composition'); plt.show()

### 4.2 Настроения, контексты, атмосфера
Слева количество треков, у которых класс получился первым, справа средняя вероятность класса. Второй график честнее, потому что трек может быть одновременно и энергичным, и радостным.

In [ ]:
def dist_plot(prefix, labels, ru, title, name):
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
    top = features[f'{prefix}_top'].value_counts().reindex(labels, fill_value=0).sort_values()
    axes[0].barh([ru[l] for l in top.index], top.values, color=ACCENT, height=0.6)
    axes[0].set_title(f'{title}: top-1 класс'); axes[0].set_xlabel('треков')
    for y, v in enumerate(top.values):
        axes[0].text(v + 0.3, y, str(v), va='center', fontsize=8, color='#52514e')
    soft = features[[f'{prefix}_{l}' for l in labels]].mean().sort_values()
    axes[1].barh([ru[c.split('_', 1)[1]] for c in soft.index], soft.values, color=ACCENT, height=0.6)
    axes[1].set_title(f'{title}: средняя вероятность'); axes[1].set_xlabel('P')
    plt.tight_layout(); savefig(name); plt.show()

dist_plot('mood', MOODS, MOOD_RU, 'Настроение', '02_mood_distribution')
dist_plot('ctx', CTXS, CTX_RU, 'Контекст', '03_context_distribution')
dist_plot('atm', ATMS, ATM_RU, 'Атмосфера', '04_atmosphere_distribution')

# сравнение с поправкой и без
cmp = pd.DataFrame({'с дебиасингом': features['mood_top'].value_counts(),
                    'без дебиасинга': features['mood_top_raw'].value_counts()}).reindex(MOODS).fillna(0).astype(int)
cmp.index = [MOOD_RU[m] for m in cmp.index]
cmp.T

### 4.3 Теги по жанрам
Средняя вероятность класса внутри жанра. Если теги работают, должно быть видно очевидное: метал агрессивный, эмбиент для сна и т. д.

In [ ]:
def genre_heatmap(prefix, labels, ru, title, name, ax):
    M = features.groupby('genre')[[f'{prefix}_{l}' for l in labels]].mean().reindex(genre_order)
    M.index = [GENRE_RU[g] for g in M.index]; M.columns = [ru[l] for l in labels]
    sns.heatmap(M, cmap='Blues', ax=ax, annot=True, fmt='.2f', annot_kws={'size': 7},
                linewidths=1, linecolor='#fcfcfb', cbar_kws={'shrink': 0.7})
    ax.set_title(title); ax.set_xlabel(''); ax.set_ylabel('')
    ax.tick_params(axis='x', rotation=40)
    for t in ax.get_xticklabels():
        t.set_ha('right')

fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))
genre_heatmap('mood', MOODS, MOOD_RU, 'Жанр × настроение', '', axes[0])
genre_heatmap('ctx', CTXS, CTX_RU, 'Жанр × контекст', '', axes[1])
genre_heatmap('atm', ATMS, ATM_RU, 'Жанр × атмосфера', '', axes[2])
plt.tight_layout(); savefig('05_genre_tag_heatmaps'); plt.show()

fig, ax = plt.subplots(figsize=(11, 5))
inst_cols = [f'inst_{i}' for i in INSTS] + ['vocals_prob']
M = features.groupby('genre')[inst_cols].mean().reindex(genre_order)
M.index = [GENRE_RU[g] for g in M.index]; M.columns = [INST_RU.get(c[5:], 'вокал') for c in inst_cols]
sns.heatmap(M, cmap='Blues', annot=True, fmt='.2f', annot_kws={'size': 7}, linewidths=1, linecolor='#fcfcfb', ax=ax, vmin=0, vmax=1)
ax.set_title('Жанр × инструменты и вокал: P(«есть»)'); ax.set_ylabel('')
plt.tight_layout(); savefig('06_genre_instruments'); plt.show()

### 4.4 Сигнальные признаки по жанрам

In [ ]:
sig_show = [('tempo_bpm', 'Темп, BPM'), ('rms_mean', 'Энергия (RMS)'), ('spectral_centroid_hz', 'Яркость, Гц'),
            ('onset_rate', 'Атак в секунду'), ('pulse_clarity', 'Ритмичность'), ('percussive_ratio', 'Доля ударных')]
fig, axes = plt.subplots(2, 3, figsize=(17, 8))
labels_ru = [GENRE_RU[g] for g in genre_order]
for ax, (col, title) in zip(axes.flat, sig_show):
    data = [features.loc[features.genre == g, col].dropna().values for g in genre_order]
    bp = ax.boxplot(data, vert=False, widths=0.55, patch_artist=True, showfliers=True,
                    medianprops={'color': '#0b0b0b', 'linewidth': 1.5},
                    boxprops={'facecolor': '#cde2fb', 'edgecolor': ACCENT},
                    whiskerprops={'color': '#9b9a95'}, capprops={'color': '#9b9a95'},
                    flierprops={'marker': 'o', 'markersize': 4, 'markerfacecolor': ACCENT, 'markeredgecolor': 'none'})
    ax.set_yticks(range(1, len(genre_order) + 1)); ax.set_yticklabels(labels_ru, fontsize=8)
    ax.invert_yaxis(); ax.set_title(title)
plt.tight_layout(); savefig('07_signal_by_genre'); plt.show()

mode_share = features.groupby('genre')['mode_major'].mean().reindex(genre_order).rename(index=GENRE_RU)
mode_share.round(2).to_frame('доля мажора').T

### 4.5 Корреляции
Беру корреляцию Спирмена: признаки распределены не нормально, и выбросов хватает.

In [ ]:
corr_cols = SIGNAL + [f'mood_{m}' for m in MOODS] + [f'ctx_{c}' for c in CTXS] + ['vocals_prob', 'year', 'log_views']
C = features[corr_cols].corr(method='spearman')
nice = {**{f'mood_{k}': f'настр: {v}' for k, v in MOOD_RU.items()}, **{f'ctx_{k}': f'конт: {v}' for k, v in CTX_RU.items()},
        'tempo_bpm': 'темп', 'rms_mean': 'энергия', 'rms_dynamic_range_db': 'дин. диапазон', 'spectral_centroid_hz': 'яркость',
        'spectral_flatness': 'шумность', 'onset_rate': 'плотн. атак', 'pulse_clarity': 'ритмичность',
        'percussive_ratio': 'доля ударных', 'zcr_mean': 'ZCR', 'mode_major': 'мажор', 'key_strength': 'сила тональности',
        'vocals_prob': 'вокал', 'year': 'год', 'log_views': 'популярность'}
Cn = C.rename(index=nice, columns=nice)
fig, ax = plt.subplots(figsize=(15, 12.5))
mask = np.triu(np.ones_like(Cn, dtype=bool), k=1)
sns.heatmap(Cn, mask=mask, cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5, linecolor='#fcfcfb',
            cbar_kws={'shrink': 0.6, 'label': 'Spearman ρ'})
ax.set_title('Корреляции признаков (Spearman)')
plt.tight_layout(); savefig('08_feature_correlations'); plt.show()

# самые сильные связи сигнал - тег
sem_cols = [f'mood_{m}' for m in MOODS] + [f'ctx_{c}' for c in CTXS] + ['vocals_prob']
pairs = C.loc[SIGNAL, sem_cols].stack().rename('rho').reset_index()
pairs.columns = ['сигнал', 'тег', 'rho']
pairs['|rho|'] = pairs['rho'].abs()
top_pairs = pairs.sort_values('|rho|', ascending=False).head(12)
top_pairs['сигнал'] = top_pairs['сигнал'].map(nice); top_pairs['тег'] = top_pairs['тег'].map(nice)
top_pairs[['сигнал', 'тег', 'rho']].round(3)

### 4.6 Жанр от CLAP против настоящего
Если модель нормально угадывает жанр, значит, и остальным её тегам можно хоть как-то доверять.

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(features['genre'], features['clapgenre_top'], labels=genre_order)
fig, ax = plt.subplots(figsize=(8.5, 7))
sns.heatmap(pd.DataFrame(cm, index=labels_ru, columns=labels_ru), annot=True, fmt='d', cmap='Blues', cbar=False,
            linewidths=1, linecolor='#fcfcfb', ax=ax)
ax.set_xlabel('предсказание CLAP (zero-shot)'); ax.set_ylabel('эталонный жанр')
acc = accuracy_score(features['genre'], features['clapgenre_top'])
ax.set_title(f'Матрица ошибок zero-shot жанра, accuracy = {acc:.2f}')
plt.xticks(rotation=40, ha='right'); plt.tight_layout(); savefig('09_zero_shot_genre_confusion'); plt.show()

per_genre_acc = (features.assign(ok=features.genre == features.clapgenre_top)
                 .groupby('genre')['ok'].mean().reindex(genre_order).rename(index=GENRE_RU))
per_genre_acc.round(2).to_frame('accuracy').T

### 4.7 Сводка по жанрам

In [ ]:
summary = []
mood_by_genre = features.groupby('genre')[[f'mood_{m}' for m in MOODS]].mean().reindex(genre_order)
ctx_by_genre = features.groupby('genre')[[f'ctx_{c}' for c in CTXS]].mean().reindex(genre_order)
for g in genre_order:
    m = mood_by_genre.loc[g].idxmax()[5:]; c = ctx_by_genre.loc[g].idxmax()[4:]
    summary.append({'жанр': GENRE_RU[g], 'типичное настроение': MOOD_RU[m], 'типичный контекст': CTX_RU[c],
                    'медианный темп': round(features.loc[features.genre == g, 'tempo_bpm'].median()),
                    'доля с вокалом': round(features.loc[features.genre == g, 'has_vocals'].mean(), 2)})
display(pd.DataFrame(summary))

# энтропия: 1 значит, что все классы встречаются одинаково часто
def norm_entropy(s, k):
    p = s.value_counts(normalize=True).values
    return float(-(p * np.log(p)).sum() / np.log(k))
print('Энтропия top-1 меток:')
for pfx, labs, name in (('mood', MOODS, 'настроение'), ('ctx', CTXS, 'контекст'), ('atm', ATMS, 'атмосфера')):
    print(f'  {name}: {norm_entropy(features[pfx + "_top"], len(labs)):.2f}, средняя уверенность top-1 = {features[pfx + "_top_conf"].mean():.2f}')
print(f'Zero-shot жанр: accuracy {acc:.2f} при случайном уровне {1/len(genre_order):.2f}')
best = top_pairs.iloc[0]
print(f'Самая сильная связь сигнал↔тег: {best["сигнал"]} ↔ {best["тег"]} (ρ = {best["rho"]:.2f})')